### Задание 1
Представьте, что вы создаёте AI-помощника для медицинского сервиса. 

Сконструируйте `Pydantic`-схему для анализа медицинских описаний рентгена грудной клетки. Каждое заключение должно включать:
- Идентификатор исследования (`study_id`).
- Список находок (`findings`). Каждая находка описывается полями: `region` (например, «правое лёгкое»), `observation` (например, «инфильтрат»), `severity` (категория: mild / moderate / severe).
- Заключение врача (`conclusion`) — строка с кратким итогом.
- Необязательные рекомендации (`recommendation`) — список текстовых элементов.


In [5]:
from pydantic import BaseModel
from enum import Enum
from typing import List, Optional
import json

class SeverityEnum(str, Enum):
    MILD = "mild"
    MODERATE = "moderate"
    SEVERE = "severe"

class Finding(BaseModel):
    region: str
    observation: str
    severity: SeverityEnum

class ChestXrayReport(BaseModel):
    study_id: str
    findings: List[Finding]
    conclusion: str
    recommendations: List[Optional[str]]

example = """
{
  "study_id": "XR12345",
  "findings": [
    {
      "region": "правое лёгкое",
      "observation": "инфильтрат",
      "severity": "moderate"
    },
    {
      "region": "левое лёгкое",
      "observation": "плевральный выпот",
      "severity": "mild"
    }
  ],
  "conclusion": "Данные за двусторонние воспалительные изменения, больше справа.",
  "recommendations": [
    "КТ грудной клетки для уточнения характера изменений",
    "Консультация пульмонолога"
  ]
}
"""

ChestXrayReport.model_validate_json(example)

ChestXrayReport(study_id='XR12345', findings=[Finding(region='правое лёгкое', observation='инфильтрат', severity=<SeverityEnum.MODERATE: 'moderate'>), Finding(region='левое лёгкое', observation='плевральный выпот', severity=<SeverityEnum.MILD: 'mild'>)], conclusion='Данные за двусторонние воспалительные изменения, больше справа.', recommendations=['КТ грудной клетки для уточнения характера изменений', 'Консультация пульмонолога'])

### Задание 2
Выберите на [OpenRouter](https://openrouter.ai/) модель, доступную бесплатно. Используя свой аккаунт, сформируйте структурированное заключение по схеме из предыдущего задания. В качестве примера используйте синтетическое описание протокола, представленное ниже.

Сначала получите результат без какого-либо описания формата.

Затем опишите формат в промпте.

Повторите оба варианта, теперь передав в качестве response_format класс, созданный в предыдущем задании.
Сравните результаты.

In [3]:
protocol = """Протокол рентгенологического исследования органов грудной клетки
Исследование: рентгенография органов грудной клетки, прямая и боковая проекции
Дата исследования: 12.09.2025
Идентификатор исследования: XR-2025-0912-001
Описание:
В правом лёгком, в нижней доле, определяется участок инфильтрации средней интенсивности размером до 4 см.
В левом лёгком, в проекции нижней доли, отмечается небольшое количество плевральной жидкости.
Сердце и корни лёгких без особенностей.
Трахея расположена по средней линии.
Заключение:
Рентгенологическая картина соответствует очагово-инфильтративным изменениям в нижней доле правого лёгкого; слева — признаки минимального плеврального выпота.
Рекомендации:
Проведение компьютерной томографии грудной клетки для уточнения характера инфильтрации.
Консультация пульмонолога.
Контрольное исследование через 10–14 дней."""

In [7]:
from openai import OpenAI

from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("OPEN_ROUTER_KEY")


client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=API_KEY,
)

# подготовьте первый вариант промпта, без описания ожидаемого формата ответа
prompt_no_format_description = f"""Выдели из представленного ниже протокола:
- "Идентификатор исследования",
- "Список находок",
- "Заключение врача",
- "Необязательные рекомендации"

Протокол:
{protocol}
"""

# подготовьте второй вариант промпта, с описанием ожидаемого формата ответа
prompt_format_description = f"""Выдели из представленного ниже протокола:
- "Идентификатор исследования",
- "Список находок",
- "Заключение врача",
- "Необязательные рекомендации"

Офроми ответ в виде json:
- study_id (str)
- findings (list). Каждая находка описывается полями:
   - region (str)
   - observation (str)
   - severity (enum: mild / moderate / severe)
- conclusion (str)
- recommendation (list(str))

Протокол:
{protocol}
"""


for prompt in [prompt_no_format_description, prompt_format_description]:
    completion = client.chat.completions.create(
      extra_body={},
      model="nvidia/nemotron-nano-9b-v2:free",  # на момент написания задания эта модель доступна бесплатно, может потребоваться найти аналог
      messages=[
        {
          "role": "user",
          "content": prompt,
        }
      ]
    )
    print(completion.choices[0].message.content)

    completion_pydantic = client.chat.completions.parse(
      extra_body={},
      model="nvidia/nemotron-nano-9b-v2:free",
      messages=[
        {
          "role": "user",
          "content": prompt,
        }
      ],
      response_format=ChestXrayReport
    )

    print(ChestXrayReport.model_validate_json(completion_pydantic.choices[0].message.content))



Вот выделенные элементы из протокола:

1. **"Идентификатор исследования":**  
   `XR-2025-0912-001`  

2. **"Список находок":**  
   - В правом лёгком, в нижней доле — участок инфильтрации средней интенсивности размером до 4 см.  
   - В левом лёгком, в нижней доле — небольшое количество плевральной жидкости.  

3. **"Заключение врача":**  
   "Рентгенологическая картина соответствует очадова-инфильтративным изменениям в нижней доле правого лёгкого; слева — признаки минимального плеврального выпота."  

4. **"Необязательные рекомендации":**  
   - Проведение компьютерной томографии грудной клетки для уточнения характера инфильтрации.  
   - Консультация пульмонолога.  
   - Контрольное исследование через 10–14 дней.

study_id='XR-2025-0912-001' findings=[Finding(region='Правёное лёгкое, нижняя полость', observation='Участок инфильтрации средней интенсивности размером до 4 см', severity=<SeverityEnum.MODERATE: 'moderate'>), Finding(region='Левёное лёгкое, нижняя доля', observation='Не